# 01. EDA and Data Preparation

This notebook performs exploratory data analysis and prepares train/validation splits for the Eye Disease Image Dataset.

## Notebook Structure

1. Environment & Path Setup  
2. Dataset Indexing  
3. Dataset Statistics  
4. Dataset Distribution & Class Imbalance  
5. Sample Visualization  
6. Corrupted Image Check  
7. Image Resolution Analysis  
8. RGB Channel Analysis  
9. Train / Validation Split  
10. Final Summary  

> Note: The dataset does not provide explicit metadata linking augmented images to their original source images.  
> Therefore, this notebook uses a stratified split and documents the limitation rather than claiming complete augmentation leakage prevention.


## 1. Environment & Path Setup

Set project paths, random seed, and output directories.

Modify `ROOT` and `DATA_ROOT` if your Google Drive path is different.


In [ ]:
import os
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.model_selection import train_test_split

# =========================
# Reproducibility
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# =========================
# Path Configuration
# =========================
ROOT = Path("/content/drive/MyDrive/eye_disease_classifier")

# Dataset root should contain class folders
DATA_ROOT = ROOT / "data" / "Combined Dataset"

# Output directories
FIGURE_DIR = ROOT / "figures" / "eda"
SPLIT_DIR = ROOT / "splits"
META_DIR = ROOT / "metadata"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("FIGURE_DIR:", FIGURE_DIR)
print("SPLIT_DIR:", SPLIT_DIR)
print("META_DIR:", META_DIR)


## 2. Dataset Indexing

Create a dataframe containing image paths, class names, file names, extensions, and numeric labels.


In [ ]:
# Supported image extensions
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

class_names = sorted([
    p.name for p in DATA_ROOT.iterdir()
    if p.is_dir()
])

print(f"Number of classes: {len(class_names)}")
print(class_names)

records = []

for label_idx, class_name in enumerate(class_names):
    class_dir = DATA_ROOT / class_name

    for img_path in sorted(class_dir.iterdir()):
        if img_path.suffix.lower() in IMG_EXTENSIONS:
            records.append({
                "image_path": str(img_path),
                "filename": img_path.name,
                "class_name": class_name,
                "label": label_idx,
                "extension": img_path.suffix.lower()
            })

df = pd.DataFrame(records)

print("Total images:", len(df))
display(df.head())

# Save metadata
data_index_path = META_DIR / "data_index.csv"
df.to_csv(data_index_path, index=False)
print("Saved:", data_index_path)


## 3. Dataset Statistics

Summarize the dataset size, number of classes, and class-wise image counts.


In [ ]:
class_counts = df["class_name"].value_counts().sort_index()

summary = {
    "total_images": len(df),
    "num_classes": df["class_name"].nunique(),
    "min_class_count": int(class_counts.min()),
    "max_class_count": int(class_counts.max()),
    "mean_class_count": float(class_counts.mean())
}

summary_df = pd.DataFrame([summary])
display(summary_df)

class_count_df = class_counts.reset_index()
class_count_df.columns = ["class_name", "count"]
display(class_count_df)

# Save statistics
summary_df.to_csv(META_DIR / "dataset_summary.csv", index=False)
class_count_df.to_csv(META_DIR / "class_distribution.csv", index=False)

print("Saved dataset summary and class distribution CSV files.")


## 4. Dataset Distribution & Class Imbalance

Visualize class-wise image counts to inspect class imbalance.

This single plot replaces separate distribution and imbalance plots to avoid redundant visualizations.


In [ ]:
# Sort by count for better imbalance visibility
plot_df = class_count_df.sort_values("count", ascending=True)

plt.figure(figsize=(10, 5))
plt.barh(plot_df["class_name"], plot_df["count"])
plt.title("Dataset Distribution & Class Imbalance", fontsize=14)
plt.xlabel("Number of Images")
plt.ylabel("Class")

for i, v in enumerate(plot_df["count"]):
    plt.text(v + max(plot_df["count"]) * 0.005, i, str(v), va="center", fontsize=8)

plt.tight_layout()

save_path = FIGURE_DIR / "dataset_distribution.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", save_path)


## 5. Sample Visualization

Display one representative sample image from each class.

This helps visually inspect the dataset and understand class-level image characteristics.


In [ ]:
sample_per_class = []

for class_name in class_names:
    class_df = df[df["class_name"] == class_name]
    sample_row = class_df.sample(1, random_state=SEED).iloc[0]
    sample_per_class.append(sample_row)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()

for ax, row in zip(axes, sample_per_class):
    img = Image.open(row["image_path"]).convert("RGB")
    ax.imshow(img)
    ax.set_title(row["class_name"], fontsize=8)
    ax.axis("off")

plt.suptitle("Representative Sample Images", fontsize=14)
plt.tight_layout()

save_path = FIGURE_DIR / "sample_images.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", save_path)


## 6. Corrupted Image Check

Check whether images can be opened successfully.

Opening every image from Google Drive can be slow, especially in Colab CPU environments.  
Therefore, this section uses a sample-based check by default.

Set `CHECK_ALL_IMAGES = True` only when you need a full dataset check.


In [ ]:
CHECK_ALL_IMAGES = False
SAMPLE_SIZE = 1000

if CHECK_ALL_IMAGES:
    check_df = df.copy()
else:
    sample_size = min(SAMPLE_SIZE, len(df))
    check_df = df.sample(sample_size, random_state=SEED).copy()

print(f"Images to check: {len(check_df)}")

corrupted_files = []

for img_path in check_df["image_path"]:
    try:
        with Image.open(img_path) as img:
            img.verify()
    except Exception as e:
        corrupted_files.append({
            "image_path": img_path,
            "error": str(e)
        })

corrupted_df = pd.DataFrame(corrupted_files)

print("Number of corrupted/unreadable images:", len(corrupted_df))

if len(corrupted_df) > 0:
    display(corrupted_df.head())
    corrupted_df.to_csv(META_DIR / "corrupted_images.csv", index=False)
    print("Saved:", META_DIR / "corrupted_images.csv")
else:
    print("No corrupted images found in the checked subset.")


## 7. Image Resolution Analysis

Inspect image width and height statistics.

A scatter plot is intentionally omitted because the images mostly share the same resolution, making the visualization less informative.


In [ ]:
# Sampling is enough for resolution inspection
RESOLUTION_SAMPLE_SIZE = min(2000, len(df))
resolution_df = df.sample(RESOLUTION_SAMPLE_SIZE, random_state=SEED).copy()

widths = []
heights = []

for img_path in resolution_df["image_path"]:
    with Image.open(img_path) as img:
        w, h = img.size
        widths.append(w)
        heights.append(h)

resolution_df["width"] = widths
resolution_df["height"] = heights
resolution_df["resolution"] = resolution_df["width"].astype(str) + "x" + resolution_df["height"].astype(str)

resolution_summary = pd.DataFrame({
    "metric": ["min_width", "max_width", "min_height", "max_height", "unique_resolution_count"],
    "value": [
        resolution_df["width"].min(),
        resolution_df["width"].max(),
        resolution_df["height"].min(),
        resolution_df["height"].max(),
        resolution_df["resolution"].nunique()
    ]
})

display(resolution_summary)

common_resolution = resolution_df["resolution"].value_counts().reset_index()
common_resolution.columns = ["resolution", "count"]
display(common_resolution.head(10))

resolution_summary.to_csv(META_DIR / "resolution_summary.csv", index=False)
common_resolution.to_csv(META_DIR / "resolution_counts.csv", index=False)

print("Saved resolution summary files.")


## 8. RGB Channel Analysis

Analyze RGB channel intensity distributions using sampled images.

This helps check brightness/color variation and supports preprocessing decisions.


In [ ]:
RGB_SAMPLE_SIZE = min(1000, len(df))
rgb_df = df.sample(RGB_SAMPLE_SIZE, random_state=SEED).copy()

r_means, g_means, b_means = [], [], []

for img_path in rgb_df["image_path"]:
    img = Image.open(img_path).convert("RGB").resize((224, 224))
    arr = np.asarray(img).astype(np.float32)

    r_means.append(arr[:, :, 0].mean())
    g_means.append(arr[:, :, 1].mean())
    b_means.append(arr[:, :, 2].mean())

rgb_stats = pd.DataFrame({
    "R_mean": r_means,
    "G_mean": g_means,
    "B_mean": b_means
})

display(rgb_stats.describe())

plt.figure(figsize=(6, 6))
plt.hist(rgb_stats["R_mean"], bins=40, alpha=0.5, label="R")
plt.hist(rgb_stats["G_mean"], bins=40, alpha=0.5, label="G")
plt.hist(rgb_stats["B_mean"], bins=40, alpha=0.5, label="B")
plt.title("RGB Channel Mean Distribution", fontsize=14)
plt.xlabel("Mean intensity")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()

save_path = FIGURE_DIR / "rgb_analysis.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

rgb_stats.to_csv(META_DIR / "rgb_channel_stats.csv", index=False)
print("Saved:", save_path)
print("Saved:", META_DIR / "rgb_channel_stats.csv")


## 9. Train / Validation Split

Create train and validation CSV files using a stratified 80:20 split.

Important limitation:

The dataset does not provide explicit metadata linking augmented images to their original source images.  
Therefore, complete prevention of augmentation leakage cannot be guaranteed.

For this reason, this notebook uses a stratified split and clearly documents the limitation.


In [ ]:
VAL_RATIO = 0.2

train_df, val_df = train_test_split(
    df,
    test_size=VAL_RATIO,
    stratify=df["label"],
    random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Validation ratio: {len(val_df) / len(df):.3f}")

print("\nTrain class distribution:")
train_dist = train_df["class_name"].value_counts().sort_index().reset_index()
train_dist.columns = ["class_name", "count"]
display(train_dist)

print("\nValidation class distribution:")
val_dist = val_df["class_name"].value_counts().sort_index().reset_index()
val_dist.columns = ["class_name", "count"]
display(val_dist)

train_path = SPLIT_DIR / "train.csv"
val_path = SPLIT_DIR / "val.csv"

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)

print("Saved:", train_path)
print("Saved:", val_path)


## 10. Final Summary

Key findings:

- The dataset contains 10 retinal disease classes.
- Severe class imbalance exists across disease categories.
- Image resolution is mostly consistent, so resolution visualization was omitted.
- RGB channel analysis shows brightness/color variation across images.
- A stratified 80:20 train/validation split was created.
- Because original-to-augmented image metadata is unavailable, complete augmentation leakage prevention cannot be guaranteed.

These findings informed preprocessing, training strategy design, and later model analysis.
